In [22]:
import requests
import pandas as pd
import time
import os
import glob
from datetime import date, timedelta

In [16]:
def buscar_temperatura_historica(df_cidades, col_lat, col_lon,
                                  data_inicio="2016-01-01", data_fim=None,
                                  tamanho_lote=10, pausa=2, max_tentativas=5,
                                  pasta_saida="temperaturas_lotes"):
    if data_fim is None:
        data_fim = (date.today() - timedelta(days=2)).isoformat()

    os.makedirs(pasta_saida, exist_ok=True)

    lats = df_cidades[col_lat].tolist()
    lons = df_cidades[col_lon].tolist()
    total_cidades = len(lats)
    total_lotes = (total_cidades - 1) // tamanho_lote + 1

    total_linhas = 0

    for i in range(0, total_cidades, tamanho_lote):
        numero_lote = i // tamanho_lote + 1
        caminho_arquivo = os.path.join(pasta_saida, f"lote_{numero_lote:05d}.csv")

        # já foi baixado numa execução anterior? pula direto.
        if os.path.exists(caminho_arquivo):
            df_existente = pd.read_csv(caminho_arquivo)
            total_linhas += len(df_existente)
            print(f"  lote {numero_lote}/{total_lotes} já existe ({len(df_existente)} linhas), pulando")
            continue

        lat_lote = lats[i:i + tamanho_lote]
        lon_lote = lons[i:i + tamanho_lote]

        params = {
            "latitude": ",".join(str(v) for v in lat_lote),
            "longitude": ",".join(str(v) for v in lon_lote),
            "start_date": data_inicio,
            "end_date": data_fim,
            "daily": "temperature_2m_max,temperature_2m_min,temperature_2m_mean",
            "timezone": "America/Sao_Paulo",
        }

        for tentativa in range(1, max_tentativas + 1):
            resp = requests.get("https://archive-api.open-meteo.com/v1/archive", params=params, timeout=60)

            if resp.status_code == 429:
                espera = int(resp.headers.get("Retry-After", pausa * tentativa * 5))
                print(f"  lote {numero_lote}/{total_lotes}: rate limit (429). "
                      f"Esperando {espera}s (tentativa {tentativa}/{max_tentativas})...")
                time.sleep(espera)
                continue

            resp.raise_for_status()
            break
        else:
            print(f"  lote {numero_lote}/{total_lotes} FALHOU após {max_tentativas} tentativas (429 repetido).\n"
                  f"  Parando aqui — o que já foi baixado está salvo em '{pasta_saida}/'.\n"
                  f"  Rode buscar_temperatura_historica(...) de novo (mesmos parâmetros) pra continuar de onde parou.")
            break

        data = resp.json()
        resultados = data if isinstance(data, list) else [data]

        partes_lote = []
        for lat_orig, lon_orig, resultado in zip(lat_lote, lon_lote, resultados):
            daily = resultado["daily"]
            partes_lote.append(pd.DataFrame({
                "latitude": lat_orig,
                "longitude": lon_orig,
                "data": daily["time"],
                "temp_max": daily["temperature_2m_max"],
                "temp_min": daily["temperature_2m_min"],
                "temp_media": daily["temperature_2m_mean"],
            }))

        df_lote = pd.concat(partes_lote, ignore_index=True)
        df_lote.to_csv(caminho_arquivo, index=False)

        total_linhas += len(df_lote)
        percentual = min(i + tamanho_lote, total_cidades) / total_cidades * 100
        print(f"  lote {numero_lote}/{total_lotes} salvo — {len(df_lote)} linhas | "
              f"{total_linhas} linhas no total | {percentual:.1f}% concluído")

        time.sleep(pausa)

    print(f"\nFim da execução. {total_linhas} linhas salvas em '{pasta_saida}/'.")


def verificar_progresso(df_cidades, tamanho_lote=10, pasta_saida="temperaturas_lotes"):
    """Mostra quantos lotes já foram baixados sem precisar rodar tudo de novo."""
    total_lotes_esperados = (len(df_cidades) - 1) // tamanho_lote + 1
    lotes_baixados = len(glob.glob(os.path.join(pasta_saida, "lote_*.csv")))
    print(f"{lotes_baixados}/{total_lotes_esperados} lotes baixados "
          f"({lotes_baixados / total_lotes_esperados * 100:.1f}%)")


In [17]:
dados_datacenters = pd.read_csv("../facilities_brazil_peering_com_lat_lng.csv")

In [20]:
dados_datacenters_sem_na = dados_datacenters.dropna(subset=["latitude", "longitude"])

In [23]:
buscar_temperatura_historica(
    dados_datacenters_sem_na,
    col_lat="latitude",
    col_lon="longitude",
    pasta_saida="temperaturas_lotes",
)

# pra checar quanto já foi sem baixar nada:
# verificar_progresso(dados_datacenters, pasta_saida="temperaturas_lotes")

  lote 1/24 salvo — 38930 linhas | 38930 linhas no total | 4.3% concluído
  lote 2/24: rate limit (429). Esperando 10s (tentativa 1/5)...
  lote 2/24: rate limit (429). Esperando 20s (tentativa 2/5)...
  lote 2/24: rate limit (429). Esperando 30s (tentativa 3/5)...
  lote 2/24: rate limit (429). Esperando 40s (tentativa 4/5)...
  lote 2/24: rate limit (429). Esperando 50s (tentativa 5/5)...
  lote 2/24 FALHOU após 5 tentativas (429 repetido).
  Parando aqui — o que já foi baixado está salvo em 'temperaturas_lotes/'.
  Rode buscar_temperatura_historica(...) de novo (mesmos parâmetros) pra continuar de onde parou.

Fim da execução. 38930 linhas salvas em 'temperaturas_lotes/'.


In [14]:
temperaturas

NameError: name 'temperaturas' is not defined